# Gold Annotation Lemma-based Sampling with Pre-filled Templates

Generates **pre-filled 100-row extraction templates** for the lemma-based
manual-extraction design, with ring-paired annotators (full overlap: both
annotators in each pair work the same 100 lemmas independently).

## What this notebook does

1. Loads the legacy pipeline's audit files (`<dict>_Parcor_audit.csv`) to
   identify which lemmas the legacy pipeline extracted from. These define the
   sampling pool.
2. Samples 100 lemmas per dict uniformly at random (fixed seed).
3. Produces, **per (annotator, dict) pair**, one pre-filled extraction template:
   - 100 rows, one per sampled lemma
   - Pre-filled: `dict_id`, `source_lemma`, `main_lemma`, `is_id_source`
   - Empty (annotator fills): `kalimat_asal`, `kalimat_tujuan`, `notes`
   - Same 100 lemmas in both annotators' templates for a given dict, in the
     same row order, so reconciliation aligns row-by-row.

> **Note:** the audit files' `lemma` column has 100% fill rate because Fix
> Parcor Batch already verifies each lemma against the prep file during its
> own lookup step. We therefore sample directly from the audit-file lemma
> space without a separate prep-file cross-reference.

## Schema of the pre-filled extraction template

| Column | Pre-filled | Filled by annotator |
|---|---|---|
| `dict_id` | ✓ | — |
| `source_lemma` | ✓ | — |
| `main_lemma` | ✓ | — |
| `is_id_source` | ✓ | — |
| `kalimat_asal` | — | ✓ |
| `kalimat_tujuan` | — | ✓ |
| `notes` | — | ✓ (required for empty rows: `"no example sentences"` or `"lemma not found in PDF"`) |

## One-pair-per-lemma policy

Template row count is fixed at 100 per dict. Each row represents one assigned
lemma. The annotator extracts at most one parallel sentence pair per row. If
the lemma has multiple example sentences in the source PDF, the annotator
picks the first one in text order (panduan §5.3). If the lemma has no
extractable pair, the annotator leaves `kalimat_asal`/`kalimat_tujuan` empty
and writes `notes = "no example sentences"`.

Annotators do not add or delete rows.

## Outputs (in `../csvAnalysis/gold_extraction/`)

- `<annotator_id>_<dict_id>_extraction.csv` × 10 (5 annotators × 2 dicts)
- `_annotator_assignment.csv` (which annotator covers which dicts)
- `_iaa_pairing.csv` (the ring pairing per dict)
- `_sampling_diagnostics.csv` (pool sizes, prep-file match rates, sampling issues)

## 1. Configuration

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# === Selected dictionaries ===
DICTS = ["91", "4", "46", "19", "24"]

# === Sample size ===
LEMMAS_PER_DICT = 100

# === Annotator ring pairing (each dict covered by 2 annotators, each annotator does 2 dicts) ===
ANNOTATOR_PAIRING = {
    "91": ("A1", "A5"),
    "4":  ("A1", "A2"),
    "46": ("A2", "A3"),
    "19": ("A3", "A4"),
    "24": ("A4", "A5"),
}

# === Source paths ===
PARCOR_DIR       = Path("../Ekstraksi/11. Parallel Corpus - Fixed")
DIRECTION_LOOKUP = Path("../Ekstraksi/LookupIsFromIndonesia.csv")

# === Output ===
OUT_DIR = Path("../csvAnalysis/gold_extraction")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

# Direction fallback (used only if LookupIsFromIndonesia.csv missing)
DIRECTION_FALLBACK = {
    "91": 0,  # Reg → Ind: Indonesian on tujuan side
    "4":  1,  # Ind → Reg: Indonesian on asal side
    "46": 0,
    "19": 1,
    "24": 0,
}

print(f"Dictionaries: {DICTS}")
print(f"Lemmas per dict: {LEMMAS_PER_DICT}")
print(f"Total extraction templates: {len(set(a for pair in ANNOTATOR_PAIRING.values() for a in pair))} annotators × 2 dicts = {2 * len(set(a for pair in ANNOTATOR_PAIRING.values() for a in pair))} templates")
print(f"\nAnnotator pairing (ring):")
for d, (x, y) in ANNOTATOR_PAIRING.items():
    print(f"  Dict #{d}: {x} & {y}")
print(f"\nOutput dir: {OUT_DIR.resolve()}")

Dictionaries: ['91', '4', '46', '19', '24']
Lemmas per dict: 100
Total extraction templates: 5 annotators × 2 dicts = 10 templates

Annotator pairing (ring):
  Dict #91: A1 & A5
  Dict #4: A1 & A2
  Dict #46: A2 & A3
  Dict #19: A3 & A4
  Dict #24: A4 & A5

Output dir: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\csvAnalysis\gold_extraction


## 2. Load direction lookup

The `is_id_source` column tells annotators which side has Indonesian text.
This is pre-filled in every template row so annotators always know which side
to transcribe in which language.

In [2]:
if DIRECTION_LOOKUP.exists():
    direction_df = pd.read_csv(DIRECTION_LOOKUP)
    direction_df["dict_id"] = direction_df["dict_id"].astype(str)
    direction_map = dict(zip(direction_df["dict_id"], direction_df["is_from_indonesia"]))
    print(f"Loaded {len(direction_map)} direction mappings from {DIRECTION_LOOKUP.name}")
else:
    print(f"⚠ Direction lookup missing at {DIRECTION_LOOKUP}, using fallback")
    direction_map = DIRECTION_FALLBACK

print("\nDirection per selected dict:")
for d in DICTS:
    if d not in direction_map:
        print(f"  Dict #{d}: ⚠ MISSING — using fallback {DIRECTION_FALLBACK.get(d, 'N/A')}")
        direction_map[d] = DIRECTION_FALLBACK.get(d, 0)
    side = "asal" if direction_map[d] == 1 else "tujuan"
    print(f"  Dict #{d}: is_id_source={direction_map[d]} (Indonesian on '{side}' side)")

⚠ Direction lookup missing at ..\Ekstraksi\LookupIsFromIndonesia.csv, using fallback

Direction per selected dict:
  Dict #91: is_id_source=0 (Indonesian on 'tujuan' side)
  Dict #4: is_id_source=1 (Indonesian on 'asal' side)
  Dict #46: is_id_source=0 (Indonesian on 'tujuan' side)
  Dict #19: is_id_source=1 (Indonesian on 'asal' side)
  Dict #24: is_id_source=0 (Indonesian on 'tujuan' side)


## 3. Load the legacy pipeline's lemma pool

Each dict's audit file has a `lemma` column listing the headword each parcor
row was extracted from. The set of distinct lemmas in this column is the
**legacy pipeline's lemma pool**: the entries the legacy pipeline successfully
extracted from.

In [3]:
legacy_lemma_pools = {}
for dict_id in DICTS:
    audit_path = PARCOR_DIR / f"{dict_id}_Parcor_audit.csv"
    if not audit_path.exists():
        print(f"⚠ Dict #{dict_id}: audit file missing at {audit_path}")
        legacy_lemma_pools[dict_id] = set()
        continue

    df = pd.read_csv(audit_path)

    if "lemma" not in df.columns:
        print(f"⚠ Dict #{dict_id}: audit file has no `lemma` column — "
              f"re-run Fix Parcor Batch with lemma lookup enabled")
        legacy_lemma_pools[dict_id] = set()
        continue

    lemmas = df["lemma"].dropna().astype(str).str.strip()
    lemmas = lemmas[lemmas != ""]
    legacy_lemma_pools[dict_id] = set(lemmas.unique())

    n_rows = len(df)
    n_lemma_filled = (df["lemma"].astype(str).str.strip() != "").sum()
    fill_rate = n_lemma_filled / n_rows if n_rows else 0
    print(f"  Dict #{dict_id}: {len(legacy_lemma_pools[dict_id])} distinct lemmas "
          f"in legacy pool (lemma fill rate {fill_rate:.1%}, "
          f"{n_rows} parcor rows total)")

  Dict #91: 4402 distinct lemmas in legacy pool (lemma fill rate 100.0%, 4762 parcor rows total)
  Dict #4: 2457 distinct lemmas in legacy pool (lemma fill rate 100.0%, 2658 parcor rows total)
  Dict #46: 5833 distinct lemmas in legacy pool (lemma fill rate 100.0%, 8484 parcor rows total)
  Dict #19: 4170 distinct lemmas in legacy pool (lemma fill rate 100.0%, 5579 parcor rows total)
  Dict #24: 2527 distinct lemmas in legacy pool (lemma fill rate 100.0%, 3518 parcor rows total)


## 4. Build the sampling pool

Verified sampling pool = legacy pipeline's lemma pool, directly.

The earlier design included a cross-reference step against the prep file
(`Pemecahan Definisi Lema/`) to verify each candidate lemma exists as a real
headword. This step has been removed because Fix Parcor Batch's lemma-lookup
process itself uses the prep file as ground truth, every lemma in the audit
file's `lemma` column was matched against `contoh_kalimat` in prep during that
process. Cross-referencing again is redundant.

Diagnostics still print pool sizes per dict for visibility.

In [4]:
verified_pools = {}
diagnostics_rows = []

for dict_id in DICTS:
    legacy = legacy_lemma_pools[dict_id]
    verified_pools[dict_id] = legacy  # accept legacy pool directly

    diagnostics_rows.append({
        "dict_id":          dict_id,
        "legacy_pool_size": len(legacy),
        "verified_pool":    len(legacy),
        "target_met":       len(legacy) >= LEMMAS_PER_DICT,
    })

    flag = "✓" if len(legacy) >= LEMMAS_PER_DICT else "⚠"
    print(f"  {flag} Dict #{dict_id}: verified pool {len(legacy)} lemmas")

diagnostics_df = pd.DataFrame(diagnostics_rows)
print("\n=== Pool diagnostics ===")
print(diagnostics_df.to_string(index=False))

short = diagnostics_df[~diagnostics_df["target_met"]]
if len(short):
    print(f"\n⚠ {len(short)} dict(s) have pool < {LEMMAS_PER_DICT} — "
          f"sampling will use the full pool for those dicts.")

  ✓ Dict #91: verified pool 4402 lemmas
  ✓ Dict #4: verified pool 2457 lemmas
  ✓ Dict #46: verified pool 5833 lemmas
  ✓ Dict #19: verified pool 4170 lemmas
  ✓ Dict #24: verified pool 2527 lemmas

=== Pool diagnostics ===
dict_id  legacy_pool_size  verified_pool  target_met
     91              4402           4402        True
      4              2457           2457        True
     46              5833           5833        True
     19              4170           4170        True
     24              2527           2527        True


## 5. Sample 100 lemmas per dict

Uniform random sampling without stratification. Fixed seed for reproducibility.
For each sampled lemma, attach `main_lemma` from the audit file (so the
annotator knows whether they're handling a main entry or sub-entry).

Rows are sorted by `source_lemma` alphabetically so the same lemma appears at
the same row position for both annotators on the dict.

In [5]:
rng = np.random.default_rng(RANDOM_SEED)

# Pre-load lemma → main_lemma mapping per dict from audit files
lemma_to_main = {}
for dict_id in DICTS:
    audit_path = PARCOR_DIR / f"{dict_id}_Parcor_audit.csv"
    if not audit_path.exists():
        lemma_to_main[dict_id] = {}
        continue
    df = pd.read_csv(audit_path)
    if "lemma" in df.columns and "main_lemma" in df.columns:
        df_valid = df[df["lemma"].notna() & (df["lemma"].astype(str).str.strip() != "")]
        mapping = (df_valid
                   .groupby("lemma")["main_lemma"]
                   .first()
                   .fillna("")
                   .astype(str)
                   .str.strip()
                   .to_dict())
        lemma_to_main[dict_id] = mapping
    else:
        lemma_to_main[dict_id] = {}


def sample_lemmas_for_dict(dict_id: str) -> pd.DataFrame:
    pool = sorted(verified_pools[dict_id])  # deterministic
    if not pool:
        return pd.DataFrame(columns=["dict_id", "source_lemma", "main_lemma", "is_id_source"])

    n_to_sample = min(LEMMAS_PER_DICT, len(pool))
    sampled_idx = rng.choice(len(pool), size=n_to_sample, replace=False)
    sampled = [pool[i] for i in sampled_idx]

    rows = []
    main_map = lemma_to_main.get(dict_id, {})
    for lem in sampled:
        main = main_map.get(lem, "")
        if not main:
            main = lem  # fallback: treat lemma as its own main
        rows.append({
            "dict_id":      dict_id,
            "source_lemma": lem,
            "main_lemma":   main,
            "is_id_source": direction_map.get(dict_id, 0),
        })

    df = pd.DataFrame(rows).sort_values("source_lemma").reset_index(drop=True)
    return df


sampled_per_dict = {}
for dict_id in DICTS:
    sampled = sample_lemmas_for_dict(dict_id)
    sampled_per_dict[dict_id] = sampled
    n_sub = (sampled["source_lemma"] != sampled["main_lemma"]).sum()
    print(f"  Dict #{dict_id}: sampled {len(sampled)} lemmas "
          f"({n_sub} sub-entries, {len(sampled) - n_sub} main entries)")

  Dict #91: sampled 100 lemmas (20 sub-entries, 80 main entries)
  Dict #4: sampled 100 lemmas (39 sub-entries, 61 main entries)
  Dict #46: sampled 100 lemmas (68 sub-entries, 32 main entries)
  Dict #19: sampled 100 lemmas (48 sub-entries, 52 main entries)
  Dict #24: sampled 100 lemmas (62 sub-entries, 38 main entries)


## 6. Write pre-filled extraction templates per (annotator, dict)

For each (annotator, dict) pair, write a CSV with 100 pre-filled rows. The
annotator's job is to fill the empty columns (`kalimat_asal`, `kalimat_tujuan`,
`notes`) without adding or deleting rows.

Both annotators on the same dict receive **the same 100 lemmas in the same row
order** so reconciliation aligns row-by-row.

In [6]:
TEMPLATE_COLUMNS = [
    "dict_id",
    "source_lemma",
    "main_lemma",
    "is_id_source",
    "kalimat_asal",   # annotator fills
    "kalimat_tujuan", # annotator fills
    "notes",          # annotator fills
]

# Build annotator → list of assigned dicts
annotator_dicts = {}
for d, (x, y) in ANNOTATOR_PAIRING.items():
    annotator_dicts.setdefault(x, []).append(d)
    annotator_dicts.setdefault(y, []).append(d)

print("=== Writing pre-filled extraction templates ===\n")
written_count = 0
for annotator_id in sorted(annotator_dicts):
    dicts_assigned = sorted(annotator_dicts[annotator_id], key=int)
    for dict_id in dicts_assigned:
        sampled = sampled_per_dict[dict_id].copy()
        # Add empty annotator-fillable columns
        sampled["kalimat_asal"]   = ""
        sampled["kalimat_tujuan"] = ""
        sampled["notes"]          = ""
        # Reorder columns to match the canonical schema
        sampled = sampled[TEMPLATE_COLUMNS]

        out_path = OUT_DIR / f"{annotator_id}_{dict_id}_extraction.csv"
        sampled.to_csv(out_path, index=False)
        written_count += 1
        print(f"  {out_path.name}: {len(sampled)} pre-filled rows")

print(f"\nWrote {written_count} templates total.")

=== Writing pre-filled extraction templates ===

  A1_4_extraction.csv: 100 pre-filled rows
  A1_91_extraction.csv: 100 pre-filled rows
  A2_4_extraction.csv: 100 pre-filled rows
  A2_46_extraction.csv: 100 pre-filled rows
  A3_19_extraction.csv: 100 pre-filled rows
  A3_46_extraction.csv: 100 pre-filled rows
  A4_19_extraction.csv: 100 pre-filled rows
  A4_24_extraction.csv: 100 pre-filled rows
  A5_24_extraction.csv: 100 pre-filled rows
  A5_91_extraction.csv: 100 pre-filled rows

Wrote 10 templates total.


## 7. Verify both annotators on each dict have identical pre-filled content

Sanity check: for each dict, the two paired annotators' templates should have
identical `source_lemma` ordering and identical pre-filled columns. If they
differ, reconciliation row-by-row alignment is broken.

In [7]:
print("=== Pair-consistency check ===\n")
all_ok = True
for dict_id, (anno_x, anno_y) in ANNOTATOR_PAIRING.items():
    path_x = OUT_DIR / f"{anno_x}_{dict_id}_extraction.csv"
    path_y = OUT_DIR / f"{anno_y}_{dict_id}_extraction.csv"
    if not path_x.exists() or not path_y.exists():
        print(f"  Dict #{dict_id}: ⚠ one or both files missing "
              f"({path_x.name}, {path_y.name})")
        all_ok = False
        continue
    df_x = pd.read_csv(path_x)
    df_y = pd.read_csv(path_y)
    # Compare pre-filled columns
    prefilled_cols = ["dict_id", "source_lemma", "main_lemma", "is_id_source"]
    same_shape = df_x.shape == df_y.shape
    same_content = df_x[prefilled_cols].equals(df_y[prefilled_cols]) if same_shape else False
    if same_shape and same_content:
        print(f"  Dict #{dict_id}: ✓ {anno_x} & {anno_y} templates match ({len(df_x)} rows)")
    else:
        print(f"  Dict #{dict_id}: ⚠ {anno_x} & {anno_y} templates differ "
              f"(shape match: {same_shape}, content match: {same_content})")
        all_ok = False

if all_ok:
    print("\n✓ All paired templates are aligned for row-by-row reconciliation.")
else:
    print("\n⚠ Some templates misaligned — investigate before distributing to annotators.")

=== Pair-consistency check ===

  Dict #91: ✓ A1 & A5 templates match (100 rows)
  Dict #4: ✓ A1 & A2 templates match (100 rows)
  Dict #46: ✓ A2 & A3 templates match (100 rows)
  Dict #19: ✓ A3 & A4 templates match (100 rows)
  Dict #24: ✓ A4 & A5 templates match (100 rows)

✓ All paired templates are aligned for row-by-row reconciliation.


## 8. Annotator assignment manifest

In [8]:
assignment_rows = []
for annotator_id in sorted(annotator_dicts):
    dicts_assigned = sorted(annotator_dicts[annotator_id], key=int)
    assignment_rows.append({
        "annotator":      annotator_id,
        "dicts_assigned": ", ".join(dicts_assigned),
        "n_dicts":        len(dicts_assigned),
        "n_lemmas_total": LEMMAS_PER_DICT * len(dicts_assigned),
        "template_files": ", ".join(f"{annotator_id}_{d}_extraction.csv" for d in dicts_assigned),
    })

assignment_df = pd.DataFrame(assignment_rows)
assignment_path = OUT_DIR / "_annotator_assignment.csv"
assignment_df.to_csv(assignment_path, index=False)

print("=== Annotator assignment ===\n")
print(assignment_df.to_string(index=False))

# Per-dict pair view
pair_rows = []
for d, (x, y) in ANNOTATOR_PAIRING.items():
    pair_rows.append({
        "dict_id":     d,
        "annotator_X": x,
        "annotator_Y": y,
        "n_lemmas":    LEMMAS_PER_DICT,
        "file_X":      f"{x}_{d}_extraction.csv",
        "file_Y":      f"{y}_{d}_extraction.csv",
    })
pair_df = pd.DataFrame(pair_rows)
pair_path = OUT_DIR / "_iaa_pairing.csv"
pair_df.to_csv(pair_path, index=False)

print("\n=== IAA pairing per dict ===\n")
print(pair_df.to_string(index=False))
print(f"\nWritten:")
print(f"  {assignment_path.name}")
print(f"  {pair_path.name}")

=== Annotator assignment ===

annotator dicts_assigned  n_dicts  n_lemmas_total                             template_files
       A1          4, 91        2             200  A1_4_extraction.csv, A1_91_extraction.csv
       A2          4, 46        2             200  A2_4_extraction.csv, A2_46_extraction.csv
       A3         19, 46        2             200 A3_19_extraction.csv, A3_46_extraction.csv
       A4         19, 24        2             200 A4_19_extraction.csv, A4_24_extraction.csv
       A5         24, 91        2             200 A5_24_extraction.csv, A5_91_extraction.csv

=== IAA pairing per dict ===

dict_id annotator_X annotator_Y  n_lemmas               file_X               file_Y
     91          A1          A5       100 A1_91_extraction.csv A5_91_extraction.csv
      4          A1          A2       100  A1_4_extraction.csv  A2_4_extraction.csv
     46          A2          A3       100 A2_46_extraction.csv A3_46_extraction.csv
     19          A3          A4       100 A3_

## 9. Sampling diagnostics

In [9]:
diagnostics_path = OUT_DIR / "_sampling_diagnostics.csv"
diagnostics_df.to_csv(diagnostics_path, index=False)

print("=== Sampling diagnostics ===\n")
print(diagnostics_df.to_string(index=False))
print(f"\nWritten: {diagnostics_path.name}")

# Final summary
print("\n=== Summary ===")
print(f"  Annotators: {len(annotator_dicts)}")
print(f"  Total templates: {sum(len(v) for v in annotator_dicts.values())}")
print(f"  Rows per template: {LEMMAS_PER_DICT}")
print(f"  Output dir: {OUT_DIR.resolve()}")

=== Sampling diagnostics ===

dict_id  legacy_pool_size  verified_pool  target_met
     91              4402           4402        True
      4              2457           2457        True
     46              5833           5833        True
     19              4170           4170        True
     24              2527           2527        True

Written: _sampling_diagnostics.csv

=== Summary ===
  Annotators: 5
  Total templates: 10
  Rows per template: 100
  Output dir: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\csvAnalysis\gold_extraction


## Run order recap

1. **Run this notebook** to produce 10 pre-filled extraction templates plus
   diagnostic and manifest files.
2. **Distribute to annotators:**
   - Each annotator receives their two pre-filled templates (per
     `_annotator_assignment.csv`).
   - Each annotator receives the source PDFs for both their assigned dicts.
   - Each annotator receives the panduan and per-dict orientation notes.
3. **Annotators work the templates**: they only fill `kalimat_asal`,
   `kalimat_tujuan`, and `notes`. They do not add or delete rows. Each row
   represents one assigned lemma; the annotator records at most one parallel
   pair, or leaves the pair columns empty with `notes = "no example sentences"`
   or `notes = "lemma not found in PDF"`.
4. **Annotators submit completed templates.**
5. **Run a separate reconciliation notebook:**
   - For each dict, load both annotators' completed templates.
   - Row-by-row reconciliation (aligned on `source_lemma`):
     - Both extracted with matching content (within similarity threshold),
       included in parcor C.
     - Both marked empty with the same notes string, recorded as agreed-no-pair.
     - All other cases, logged in disagreement file for researcher review.
   - Compute Cohen's κ on the extract/skip decision per dict.
   - Output `parcor_C/<dict_id>_final.csv` and `<dict_id>_disagreement.csv`.
6. **Run pipeline-comparison notebook(s):**
   A vs C, B vs C (when ready), A1 vs C, B1 vs C.